In [6]:
import pandas as pd
import numpy as np

In [2]:
pip install pandas

  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\adity\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [7]:
pd.set_option('display.max_columns', None)


In [8]:
orders = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_orders_dataset.csv")
customers = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_customers_dataset.csv")
order_items = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_order_items_dataset.csv")
products = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_products_dataset.csv")
payments = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_order_payments_dataset.csv")
reviews = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_order_reviews_dataset.csv")
sellers = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_sellers_dataset.csv")
geolocation = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\olist_geolocation_dataset.csv")
categories = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Raw\product_category_name_translation.csv")

In [9]:
print("Orders:",orders.shape)
print("Customers:",customers.shape)
print("Order_items:",order_items.shape)
print("products:",products.shape)
print("payments:",payments.shape)
print("reviews:",reviews.shape)
print("sellers:",sellers.shape)
print("geolocation:",geolocation.shape)
print("categories:",categories.shape)

Orders: (99441, 8)
Customers: (99441, 5)
Order_items: (112650, 7)
products: (32951, 9)
payments: (103886, 5)
reviews: (99224, 7)
sellers: (3095, 4)
geolocation: (1000163, 5)
categories: (71, 2)


In [10]:
def clean_columns(df):
    df.columns = df.columns.str.lower().str.strip()
    return df

orders = clean_columns(orders)
customers = clean_columns(customers)
order_items = clean_columns(order_items)
products = clean_columns(products)
payments = clean_columns(payments)
reviews = clean_columns(reviews)
sellers = clean_columns(sellers)
geolocation = clean_columns(geolocation)
categories = clean_columns(categories)

In [11]:
print(orders.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')


In [12]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [13]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [14]:
# Delivery days
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

# Delay days
orders["delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

# Late delivery flag
orders["is_late"] = orders["delay_days"] > 0
orders["is_late"] = orders["is_late"].astype(int)

orders[["delivery_days", "delay_days", "is_late"]].head()

,delivery_days,delay_days,is_late
0,8.0,-8.0,0
1,13.0,-6.0,0
2,9.0,-18.0,0
3,13.0,-13.0,0
4,2.0,-10.0,0


In [15]:
# Merge price into orders
order_revenue = order_items.groupby("order_id")["price"].sum().reset_index()
order_revenue.rename(columns={"price": "revenue"}, inplace=True)

orders = orders.merge(order_revenue, on="order_id", how="left")

# Simulate cost as 70% of revenue
orders["cost"] = orders["revenue"] * 0.7

# Profit
orders["profit"] = orders["revenue"] - orders["cost"]

# Profit margin
orders["profit_margin"] = orders["profit"] / orders["revenue"]

orders[["revenue", "cost", "profit", "profit_margin"]].head()

,revenue,cost,profit,profit_margin
0,29.99,20.993,8.997,0.3
1,118.70,83.090,35.610,0.3
2,159.90,111.930,47.970,0.3
3,45.00,31.500,13.500,0.3
4,19.90,13.930,5.970,0.3


In [16]:
# Create product-level cost ratio based on price
order_items["cost_ratio"] = np.where(
    order_items["price"] < 50, 0.6,
    np.where(order_items["price"] < 200, 0.7, 0.8)
)

# Calculate item cost
order_items["item_cost"] = order_items["price"] * order_items["cost_ratio"]

# Aggregate revenue and cost per order
order_finance = order_items.groupby("order_id").agg(
    revenue=("price", "sum"),
    cost=("item_cost", "sum")
).reset_index()

In [17]:
# Merge with orders
orders = orders.drop(columns=["revenue", "cost", "profit", "profit_margin"], errors="ignore")
orders = orders.merge(order_finance, on="order_id", how="left")


In [18]:
# Recalculate profit
orders["profit"] = orders["revenue"] - orders["cost"]
orders["profit_margin"] = orders["profit"] / orders["revenue"]

orders[["revenue", "cost", "profit", "profit_margin"]].head()

,revenue,cost,profit,profit_margin
0,29.99,17.994,11.996,0.4
1,118.70,83.090,35.610,0.3
2,159.90,111.930,47.970,0.3
3,45.00,27.000,18.000,0.4
4,19.90,11.940,7.960,0.4


In [21]:
orders.to_csv("C:/Users/adity/ecommerce-sales-profitability-analysis/Data/cleaned/orders_cleaned.csv", index=False)

In [20]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,cost_ratio,item_cost
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,0.7,41.230
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,0.8,191.920
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,0.7,139.300
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,0.6,7.794
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,0.7,139.930


In [22]:
order_items.columns = order_items.columns.str.lower().str.strip()
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'cost_ratio',
       'item_cost'],
      dtype='object')

In [23]:
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
cost_ratio             0
item_cost              0
dtype: int64

In [24]:
order_items = order_items[order_items["price"] > 0]
order_items = order_items[order_items["freight_value"] >= 0]

In [25]:
order_items["price"] = order_items["price"].astype(float)
order_items["freight_value"] = order_items["freight_value"].astype(float)

In [28]:
order_items = order_items[[
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
]]

In [29]:
order_items.to_csv("C:/Users/adity/ecommerce-sales-profitability-analysis/Data/cleaned/order_items_cleaned.csv", index=False)

In [30]:
categories.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [31]:
categories.columns = categories.columns.str.lower().str.strip()
categories.columns


Index(['product_category_name', 'product_category_name_english'], dtype='object')

In [32]:
categories.isnull().sum()


product_category_name            0
product_category_name_english    0
dtype: int64

In [33]:
categories.to_csv("C:/Users/adity/ecommerce-sales-profitability-analysis/Data/cleaned/categories_cleaned.csv", index=False)


In [34]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [35]:
products.columns = products.columns.str.lower().str.strip()
products.columns


Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

In [36]:
products_cleaned = products[[
    "product_id",
    "product_category_name"
]]


In [37]:
products_cleaned.to_csv("C:/Users/adity/ecommerce-sales-profitability-analysis/Data/cleaned/products_cleaned.csv", index=False)

In [39]:
orders = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Cleaned\orders_cleaned.csv")
order_items = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Cleaned\order_items_cleaned.csv")
products = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Cleaned\products_cleaned.csv")
categories = pd.read_csv(r"C:\Users\adity\ecommerce-sales-profitability-analysis\Data\Cleaned\categories_cleaned.csv")

In [40]:
order_items_merged = order_items.merge(
    orders[["order_id", "revenue", "profit"]],
    on="order_id",
    how="inner"
)

In [41]:
order_items_merged["product_profit"] = (
    order_items_merged["price"] / order_items_merged["revenue"]
) * order_items_merged["profit"]

In [42]:
product_category = products.merge(
    categories,
    on="product_category_name",
    how="left"
)

order_items_full = order_items_merged.merge(
    product_category,
    on="product_id",
    how="left"
)

In [43]:
category_profitability = (
    order_items_full
    .groupby("product_category_name_english")
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("price", "sum"),
        total_profit=("product_profit", "sum")
    )
    .reset_index()
)

category_profitability["profit_margin_pct"] = (
    category_profitability["total_profit"]
    / category_profitability["total_revenue"]
) * 100

category_profitability.sort_values(
    by="total_profit", ascending=False
).head(10)

,product_category_name_english,total_orders,total_revenue,total_profit,profit_margin_pct
43,health_beauty,8836,1258681.34,322612.059458,25.630956
7,bed_bath_table,9417,1036988.68,301372.254240,29.062251
70,watches_gifts,5624,1205005.68,289524.555519,24.026821
65,sports_leisure,7720,988048.97,268461.634878,27.170884
15,computers_accessories,6689,911954.32,250196.259124,27.435175
39,furniture_decor,6449,729762.49,211271.114981,28.950668
49,housewares,5884,632248.66,179711.629216,28.424201
20,cool_stuff,3632,635290.85,159027.508280,25.032237
5,auto,3897,592720.11,151221.990057,25.513221
42,garden_tools,3518,485256.46,130380.958952,26.868464


In [44]:
category_profitability.to_csv("C:/Users/adity/ecommerce-sales-profitability-analysis/Data/cleaned/category_profitability.csv",
    index=False
)
